# BioSkills Lab — Chapter 8
## Logistic Regression and Random Forests on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> This notebook is self-contained. It will load TCGA data automatically if Chapter 7 variables are not already in memory.

In [ ]:
!pip install -q scikit-learn numpy pandas matplotlib requests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
print('Libraries loaded')

In [ ]:
# Load TCGA data via cBioPortal API (same as Chapter 7)
# Skip this cell if you already ran Chapter 7 in the same session
try:
    _ = X_train_pca
    print('Chapter 7 data already loaded — skipping download')
except NameError:
    print('Loading TCGA data from cBioPortal...')
    BASE = 'https://www.cbioportal.org/api'
    STUDIES = {'BRCA':'brca_tcga','LUAD':'luad_tcga','KIRC':'kirc_tcga','PRAD':'prad_tcga','COAD':'coad_tcga'}
    all_frames = []
    for cancer, study in STUDIES.items():
        r = requests.get(f'{BASE}/studies/{study}/samples', timeout=30)
        samples = [s['sampleId'] for s in r.json()][:150]
        profile = f'{study}_mrna'
        r2 = requests.post(f'{BASE}/molecular-profiles/{profile}/molecular-data/fetch',
                           json={'sampleIds': samples}, params={'projection': 'SUMMARY'}, timeout=120)
        if r2.status_code != 200:
            continue
        df = pd.DataFrame([{'sample': d['sampleId'], 'gene': d['entrezGeneId'], 'value': d['value']}
                           for d in r2.json() if d['value'] is not None])
        if df.empty: continue
        pivot = df.pivot(index='sample', columns='gene', values='value')
        pivot['cancer_type'] = cancer
        all_frames.append(pivot)
        print(f'  {cancer}: {len(pivot)} samples')
    if not all_frames:
        print('API unavailable — using synthetic data')
        np.random.seed(42)
        n, g = 150, 3000
        cancer_types = ['BRCA','LUAD','KIRC','PRAD','COAD']
        blocks, labels_list = [], []
        for i, ct in enumerate(cancer_types):
            X = np.random.randn(n, g) * 0.5
            X[:, i*600:(i+1)*600] += 3
            blocks.append(X)
            labels_list.extend([ct]*n)
        expr = pd.DataFrame(np.vstack(blocks), columns=[f'g{i}' for i in range(g)])
        expr['cancer_type'] = labels_list
    else:
        expr = pd.concat(all_frames)
    y_raw = expr['cancer_type'].values
    X_raw = expr.drop(columns=['cancer_type']).values.astype(np.float32)
    X_raw = np.nan_to_num(X_raw, nan=0.0)
    le = LabelEncoder()
    y_enc = le.fit_transform(y_raw)
    X_train, X_test, y_train, y_test = train_test_split(X_raw, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    pca = PCA(n_components=min(50, X_train_s.shape[1]-1))
    X_train_pca = pca.fit_transform(X_train_s)
    X_test_pca  = pca.transform(X_test_s)
    print(f'Data ready. PCA shape: {X_train_pca.shape}')

## Logistic Regression

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_pca, y_train)
y_pred_lr = lr.predict(X_test_pca)
print(f'LR Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}')
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False)
plt.title('Logistic Regression Confusion Matrix')
plt.tight_layout(); plt.show()

## Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_features='sqrt', n_jobs=-1, random_state=42)
rf.fit(X_train_pca, y_train)
y_pred_rf = rf.predict(X_test_pca)
print(f'RF Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}')

In [ ]:
importances = rf.feature_importances_
plt.figure(figsize=(10,4))
plt.bar(range(len(importances)), importances)
plt.xlabel('Principal Component'); plt.ylabel('Feature Importance')
plt.title('Random Forest Feature Importances')
plt.show()

In [ ]:
print('Method              Accuracy')
print(f'Logistic Regression {accuracy_score(y_test, y_pred_lr):.3f}')
print(f'Random Forest       {accuracy_score(y_test, y_pred_rf):.3f}')